In [ ]:
# BO for first pass exo work
# modify run_vellanky
# take in scalar y and vector (parameters) x

import json
import os
 
import numpy as np
from scipy.optimize import minimize
from scipy.spatial.distance import cdist
from scipy.stats import norm

input_file = "input.json"      # whatever this needs to be to connect to exo
output_file = "output.json"    # will head right back to exo
# my understanding is the first time we do this (today) it will just be number in number out (?)
history_file = "history.json"  # tbh i don't know if this is going to work the way i want it to because idk how it works

bounds = (-1.0, 1.0)   # bounds for search, will need to change these to something reasonable?
ell = 0.1             # kernel lengthscale, i put this in as a set value but we could totally use loglikelihood here as well
noise = 1e-6          # noise for mtx
nrestarts = 20        # restarts for optimizing
minimize_target = True  # in case we need to maximize this is here 2 be changed

# right so this is the kernel that we can change
def kernel(X1, X2, ell):
    D = cdist(X1, X2, metric="euclidean")
    return np.exp(-(D**2) / (2 * ell))

# i dont know if im using classes right davide HELP
class GaussianProcess:
    # we can initialize the GP
    def __init__(self, ell=ell, noise=noise):
        self.ell = ell
        self.noise = noise
    # fit the GP
    def fit(self, X, y):
        self.X = X
        self.y = y
        K = kernel(X, X, self.ell) + self.noise * np.eye(len(X))
        # cholesky
        Lk = np.linalg.cholesky(K)
        self.K_inv = np.linalg.inv(Lk @ Lk.T)

        return self
    # predict mu and sigma
    def predict(self, X_star):
        
        SX = kernel(X_star, self.X, self.ell) #SX
        SXX = kernel(X_star, X_star, self.ell) #SXX
        
        mu = SX @ self.K_inv @ self.y
        cov = SXX - SX @ self.K_inv @ SX.T
        var = np.clip(np.diag(cov), 1e-12, None)

        # cholesky not fully in bc idk if it's right
        # Sigma = kernel(self.X, self.X, self.ell) + self.noise * np.eye(len(self.X))
        # L = np.linalg.cholesky(Sigma) # take choelsky decomp of sigma
        # alpha = solve_triangular(L.T, solve_triangular(L, y, lower=True), lower=False) # solve it yay
        # mu = SX @ alpha
        # V = solve_triangular(L, SX.T, lower=True)
        # var = SXX - V.T @ V # sigma, that V.T @ V is replacement of SX @ Sigma_inv @ SX.T because of choelsky

        return mu, var
# how do we see if new estimate is good? expected improvement
def expected_improvement(x, gp, y_best): # computa help in this whole function. sadly.
    x = np.atleast_2d(x) # got computa help here to make sure the numbers work ok bc they need to be in a 2d array and we might get smth like [,5]
    mu, var = gp.predict(x) # our predict that we wrote
    sigma = np.sqrt(var)
    with np.errstate(divide="ignore"): # if pts similar we might get sigma=0 so this stops us from getting that error
        z = (y_best - mu) / sigma
        ei = (y_best - mu) * norm.cdf(z) + sigma * norm.pdf(z)
        ei = np.where(sigma == 0.0, 0.0, ei) # even tho we deal with the error here i think
    return ei[0]
# how do we pick the next point of the n points we try
def propose_next_point(gp, y_best, bounds, n_restarts=nrestarts, rng=None):
    rng = rng or np.random.default_rng() # we will prob need to change this
    best_x, best_val = None, -np.inf
    for x0 in rng.uniform(bounds[0], bounds[1], size=(n_restarts, 1)): # how we loop
        res = minimize(
            lambda x: -expected_improvement(x, gp, y_best),
            x0=x0,
            bounds=[bounds],
        )
        val = -res.fun
        if val > best_val: # if we get a better value, update the best one
            best_val = val
            best_x = res.x
    return float(best_x[0])
# straight up u guys i have no idea if this is going to work
# computa help here i dont know json at all
# obviously we need this though
def load_history():
    if os.path.exists(history_file):
        with open(history_file) as f:
            data = json.load(f)
        return data.get("X", []), data.get("y", []), data.get("last_x", None)
    return [], [], None
 
 
def save_history(X, y, last_x):
    with open(history_file, "w") as f:
        json.dump({"X": X, "y": y, "last_x": last_x}, f)

# ok now here is how we run the whole thing
def main():
    # open the input

    # search for latest run number, save both x and y from that run
    # instead of scalar, get last two columns (L and R motors) and find area under the curve in this code
    # param rn gain of 3

    with open(input_file) as f:
        f = json.load(f)
        y_new = f["y"] 
    # check our history
    X_list, y_list, last_x = load_history()
 
    # the value we just read corresponds to the point we proposed last time
    # and we can now add to a list if we have new info
    if last_x is not None:
        X_list.append(last_x)
        y_list.append(y_new if minimize_target else -y_new)
 
    if len(X_list) == 0:
        # if no observations, we start with a random point
        next_x = float(np.random.default_rng().uniform(*bounds))
    else: # yay gp!!!!!!
        X = np.array(X_list).reshape(-1, 1)
        y = np.array(y_list)
        gp = GaussianProcess().fit(X, y)
        y_best = y.min()
        next_x = propose_next_point(gp, y_best, bounds)
    # add into the history file
    save_history(X_list, y_list, next_x)
    # write output
    with open(output_file, "w") as f:
        f.write(str(next_x))
 
# just learned this today this is how we make sure this runs only when we ask
if __name__ == "__main__":
    main()